# Create connectome data table

In this notebook, we create a connectome data table that can be used to visualize the distances between embeddings for various foundation models. We have 9 foundation models, where we have pre-extracted embeddings from tumor bounding boxes for the NLST collection. These bounding boxes are from Sybil.

Note:
- Currently the pre-extracted embeddings are stored in Google Drive. Later, we will include them in a GitHub repo, where they can be publicly accessed.

Deepa Krishnaswamy

Brigham and Women's Hospital

March 2026

# Parameterization

In [1]:
#@title Enter your Project ID here
# initialize this variable with your Google Cloud Project ID!
project_name = "idc-external-018" #@param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = project_name

!gcloud config set project $project_name

from google.colab import auth
auth.authenticate_user()

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



# Setup

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [6]:
# Later, this can be replaced by GitHub release attachments

MAIN_DIR = "/content/gdrive/MyDrive/Colab Notebooks/nlst-cbir-connectome/features"

PROJECT_ID = project_name
DATASET_ID = "nlst_cbir_connectome"

In [4]:
!ls "$MAIN_DIR"

CTClipVit_features.pkl	Merlin_features.pkl	SUPREME_features.pkl
CTFM_features.pkl	ModelsGen_features.pkl	VISTA3D_features.pkl
FMCIB_features.pkl	PASTA_features.pkl	Voco_features.pkl


# Create dataset

In [7]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"
dataset = client.create_dataset(dataset, exists_ok=True)

print(f"Dataset {DATASET_ID} is ready.")

Dataset nlst_cbir_connectome is ready.


# Create feature BQ tables


In [8]:
import pickle
import pandas as pd

MODEL_LIST = ["CTClipVit", "CTFM", "FMCIB", "Merlin", "ModelsGen", "PASTA", "SUPREME", "VISTA3D", "Voco"]

def process_and_upload(model_name):
    path = os.path.join(MAIN_DIR, f"{model_name}_features.pkl")
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return

    with open(path, 'rb') as f:
        data = pickle.load(f)

    rows = []
    for item in data['all']:
        meta = item['row']
        rows.append({
            "model": model_name,
            "PatientID": meta.get("PatientID"),
            "SeriesInstanceUID": meta.get("SeriesInstanceUID"),
            "SOPInstanceUID": meta.get("SOPInstanceUID"),
            "image_path": meta.get("image_path"),
            "coordX": meta.get("coordX"),
            "coordY": meta.get("coordY"),
            "coordZ": meta.get("coordZ"),
            "label_stage": meta.get("labels_de_stag_mapped"),
            "label_type": meta.get("labels_de_type"),
            "label_type_mapped": meta.get("labels_de_type_mapped"),
            "embedding": item['feature'].squeeze().tolist()
        })

    df = pd.DataFrame(rows)
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{model_name}_features"

    # Configure schema for the ARRAY<FLOAT64>
    job_config = bigquery.LoadJobConfig(
        schema=[bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED")],
        write_disposition="WRITE_TRUNCATE",
    )

    client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
    print(f"Uploaded {model_name}")

for model in MODEL_LIST:
    process_and_upload(model)

Uploaded CTClipVit
Uploaded CTFM
Uploaded FMCIB
Uploaded Merlin
Uploaded ModelsGen
Uploaded PASTA
Uploaded SUPREME
Uploaded VISTA3D
Uploaded Voco


# Create combined distances table

In [9]:
# 1. Calculate individual distances (Optimized: No URLs, No Joins)
for model in MODEL_LIST:
    source = f"{PROJECT_ID}.{DATASET_ID}.{model}_features"
    dest = f"{PROJECT_ID}.{DATASET_ID}.{model}_distances"

    # We removed the CONCAT lines and the JOIN to bigquery-public-data.
    # Since all the metadata we need (PatientID, labels) is already in your
    # features table, we can just query your data directly.
    query = f"""
    CREATE OR REPLACE TABLE `{dest}` AS
    WITH single_lesion AS (
        SELECT PatientID FROM `{source}`
        GROUP BY PatientID
        HAVING COUNT(SOPInstanceUID) = 1
    ),
    pts AS (
        SELECT f.* FROM `{source}` f
        JOIN single_lesion s ON f.PatientID = s.PatientID
    )
    SELECT
        "{model}" AS model_name,
        -- Query Patient Info
        p1.PatientID AS q_PatientID,
        p1.label_type_mapped AS q_type,
        p1.label_stage AS q_stage,

        -- Match Patient Info
        p2.PatientID AS m_PatientID,
        p2.label_type_mapped AS m_type,
        p2.label_stage AS m_stage,

        -- Distance Computation
        ML.DISTANCE(p1.embedding, p2.embedding, 'COSINE') AS distance,
        # (1 - ML.DISTANCE(p1.embedding, p2.embedding, 'COSINE')) * 100 AS similarity_pct
        GREATEST(0, (1 - ML.DISTANCE(p1.embedding, p2.embedding, 'COSINE'))) * 100 AS similarity_pct

    FROM pts p1
    CROSS JOIN pts p2
    WHERE p1.PatientID < p2.PatientID
    """
    client.query(query).result()
    print(f"Distances computed for {model}")

# 2. Final Union
# This stacks all 9 model results into one large table for visualization
union_sql = " UNION ALL ".join([f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{m}_distances`" for m in MODEL_LIST])
client.query(f"CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.all_model_distances` AS {union_sql}").result()

print("Success! Final consolidated table 'all_model_distances' created.")

Distances computed for CTClipVit
Distances computed for CTFM
Distances computed for FMCIB
Distances computed for Merlin
Distances computed for ModelsGen
Distances computed for PASTA
Distances computed for SUPREME
Distances computed for VISTA3D
Distances computed for Voco
Success! Final consolidated table 'all_model_distances' created.


# Create a separate table with the UIDs

In [10]:
# PatientID
# StudyInstanceUID
# SeriesInstanceUID of CT
# SeriesInstanceUID of Sybil bounding box SR


In [11]:
# Define the table names
output_table = f"{PROJECT_ID}.{DATASET_ID}.patient_sr_mapping"
feature_table = f"{PROJECT_ID}.{DATASET_ID}.CTClipVit_features" # Using CTClipVit as the reference cohort

query = f"""
CREATE OR REPLACE TABLE `{output_table}` AS
WITH select_single_patients AS (
  /* Identify patients that only have one entry in the features table */
  SELECT
    PatientID,
    SeriesInstanceUID AS SeriesInstanceUID_Reference
  FROM
    `{feature_table}`
  WHERE PatientID IN (
    SELECT PatientID
    FROM `{feature_table}`
    GROUP BY PatientID
    HAVING COUNT(*) = 1
  )
)

SELECT DISTINCT
  s.PatientID,
  d.StudyInstanceUID,
  s.SeriesInstanceUID_Reference AS SeriesInstanceUID_reference,
  d.SeriesInstanceUID           AS SeriesInstanceUID_sr
FROM
  select_single_patients AS s
JOIN
  `bigquery-public-data.idc_current.dicom_all` AS d
  ON d.Modality = 'SR'
 AND d.collection_id = 'nlst'
 AND d.SeriesDescription LIKE "%Sybil%"
/* Nested Unnesting to reach the referenced SeriesInstanceUID in SR files */
JOIN
  UNNEST(d.CurrentRequestedProcedureEvidenceSequence) AS evidence
JOIN
  UNNEST(evidence.ReferencedSeriesSequence) AS ref
  ON ref.SeriesInstanceUID = s.SeriesInstanceUID_Reference
ORDER BY
  PatientID
"""

print(f"Running query to create {output_table}...")
client.query(query).result()
print("Success! Mapping table created.")

Running query to create idc-external-018.nlst_cbir_connectome.patient_sr_mapping...
Success! Mapping table created.


# Create the final connectome data table

In [12]:
# Now join the all_model_distances table with the patient_sr_mapping table in order to get the
# final connectome_data table.
# This will also include the OHIF series viewer urls.

In [13]:
# Define the final table name
final_table_id = f"{PROJECT_ID}.{DATASET_ID}.connectome_data"

# This query links your distances to the SR mappings for both patients (q and m)
query = f"""
CREATE OR REPLACE TABLE `{final_table_id}` AS
SELECT
  d.*,
  -- Query patient viewer URL: Combines CT Series and Sybil SR Series
  IF(
    qmap.StudyInstanceUID IS NOT NULL,
    CONCAT(
      'https://viewer.imaging.datacommons.cancer.gov/v3/viewer/?StudyInstanceUIDs=',
      qmap.StudyInstanceUID,
      '&SeriesInstanceUIDs=',
      qmap.SeriesInstanceUID_Reference,
      ',',
      qmap.SeriesInstanceUID_sr
    ),
    NULL
  ) AS q_viewer_url_series,
  -- Matched patient viewer URL: Combines CT Series and Sybil SR Series
  IF(
    mmap.StudyInstanceUID IS NOT NULL,
    CONCAT(
      'https://viewer.imaging.datacommons.cancer.gov/v3/viewer/?StudyInstanceUIDs=',
      mmap.StudyInstanceUID,
      '&SeriesInstanceUIDs=',
      mmap.SeriesInstanceUID_Reference,
      ',',
      mmap.SeriesInstanceUID_sr
    ),
    NULL
  ) AS m_viewer_url_series
FROM `{PROJECT_ID}.{DATASET_ID}.all_model_distances` AS d
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.patient_sr_mapping` AS qmap
  ON d.q_PatientID = qmap.PatientID
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.patient_sr_mapping` AS mmap
  ON d.m_PatientID = mmap.PatientID;
"""

print(f"Generating final connectome table: {final_table_id}...")
client.query(query).result()
print("Success! The final connectome table with OHIF viewer links is now ready.")

Generating final connectome table: idc-external-018.nlst_cbir_connectome.connectome_data...
Success! The final connectome table with OHIF viewer links is now ready.
